# Módulo 03 — Construcción del índice semántico

## 1. Objetivo del módulo

Este módulo construye el **índice semántico vectorial** que utilizará TechMind para realizar búsquedas de documentos relacionados por significado.

En los módulos anteriores se preparó el dataset multilingüe y se generaron embeddings para los documentos utilizando `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`. El Módulo 02 representa cada documento mediante un vector numérico de 384 dimensiones.

El objetivo aquí es tomar esos vectores y construir una estructura que permita encontrar los documentos cuyos embeddings están más cerca de un embedding de consulta.

## 2. ¿Qué problema resuelve?

Una comparación literal de palabras no es suficiente para relacionar textos escritos en inglés, español y portugués. Por ejemplo, `Impacto de la inteligencia artificial`, `Effects of artificial intelligence` e `Impacto da inteligência artificial` pueden expresar conceptos similares aunque utilicen palabras distintas.

Los embeddings transforman esos textos en puntos dentro de un espacio vectorial. Este módulo construye la estructura necesaria para comparar sus posiciones utilizando **distancia coseno**.

```text
Documento → Embedding → Vector de 384 dimensiones → Índice → Vecinos cercanos
```

## 3. Entrada del módulo

La entrada principal es `models/embeddings.npy`, generado por el Módulo 02. Cada fila representa un documento y cada fila contiene 384 valores. Para el prototipo se espera la forma `(900, 384)`: 900 documentos por 384 valores.

El orden es fundamental: la posición `i` del embedding debe corresponder al documento que ocupa la posición `i` en `documents.json` o `dataset_procesado.json`. El índice devuelve posiciones, que luego se utilizan para recuperar los metadatos del documento.

## 4. Configuración del índice

Se utiliza `NearestNeighbors(metric='cosine', algorithm='brute')`. La distancia coseno mide el ángulo entre vectores y permite comparar orientación semántica, independientemente de su magnitud. Como los embeddings están normalizados, esta comparación es especialmente adecuada.

`algorithm='brute'` compara la consulta contra todos los vectores. Es una opción simple, determinista y fácil de explicar para el prototipo, aunque su coste crece con el número de documentos. En una versión de producción podrían evaluarse índices aproximados como FAISS, Annoy o HNSW.

## 5. Importante: `fit()` no es entrenamiento supervisado

La llamada `semantic_index.fit(embeddings)` no aprende clases, etiquetas ni categorías. `NearestNeighbors` no recibe una variable objetivo y no realiza clasificación. `fit()` solamente almacena o estructura los vectores para poder buscar vecinos cuando llegue una consulta.

## 6. Salida del módulo

El resultado principal es `models/semantic_index.joblib`, que contiene el objeto `NearestNeighbors` ya ajustado. El Módulo 04 cargará este archivo, codificará una consulta y solicitará sus vecinos más cercanos.

## 7. Relación con el siguiente módulo

El Módulo 04 utilizará el índice siguiendo este flujo: consulta textual → embedding de consulta → vecinos cercanos → distancia coseno → similitud → documentos relacionados. La similitud se obtiene transformando la distancia como `similitud = 1 - distancia`.

## 8. Limitaciones

Este índice no modifica los embeddings ni mejora el modelo. Su calidad depende del modelo preentrenado del Módulo 02 y de la calidad de los textos. Además, la estrategia brute-force puede ser lenta y consumir más recursos cuando el corpus crezca significativamente.

In [1]:
import sys
from pathlib import Path
import numpy as np
import joblib
sys.path.insert(0, str(Path.cwd() / 'src'))
from techmind.config import get_paths
from techmind.persistence.artifacts import load_embeddings, load_documents
from techmind.search.semantic_search import build_index
paths = get_paths(); embeddings = load_embeddings(paths.embeddings)
index = build_index(embeddings); joblib.dump(index, paths.index)
print('[TechMind] Semantic index persisted.')

[TechMind] Semantic index persisted.
